In [1]:
import sys
import os
from preprocess import MinMaxNormalizer
from keras.models import load_model
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('variational_ae.py'), '..')))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('encoder.py'), '..')))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('decoder.py'), '..')))
from variational_ae import VariationalAutoencoder
from encoder import EncoderBuilder, Sampling
from decoder import DecoderBuilder
import librosa
import h5py
import numpy as np
import soundfile as sf

2025-08-20 01:35:36.607450: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-20 01:35:36.652801: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-20 01:35:37.914887: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
class SoundGenerator:
    def __init__(self, vae, hop_length, sample_rate):
        self.vae = vae
        self.hop_length = hop_length
        self.sample_rate = sample_rate
        self._min_max_normalizer = MinMaxNormalizer()
        
    def generate(self, spectrograms, spectrogram_type):
        generated_spectrograms = self.vae.predict(spectrograms)
        signals = self.convert_spectrograms_to_audio(generated_spectrograms, spectrogram_type)
        return signals

    def convert_spectrograms_to_audio(self, spectrograms, spectrogram_type):
        signals = []
        spectrograms = np.squeeze(spectrograms, axis=-1)
        for spectrogram in spectrograms:
            denormalized_spectrogram = self._min_max_normalizer.denormalize(spectrogram)
            if spectrogram_type == "log_mel_spec":
                power_spec = librosa.db_to_power(denormalized_spectrogram, ref=1.0)
                signal = librosa.feature.inverse.mel_to_audio(
                    power_spec, sr=self.sample_rate, hop_length=self.hop_length
                )
            elif spectrogram_type == "log_spec":
                amp_spec = librosa.db_to_amplitude(denormalized_spectrogram, ref=1.0)
                signal = librosa.istft(amp_spec, hop_length=self.hop_length)
            else:
                raise ValueError(f"Unsupported spectrogram_type: {spectrogram_type}")
            signals.append(signal)
        return signals

In [3]:
with h5py.File('Dataset/log_mel_spec_data_dataset.h5', 'r') as h5f:
    log_mel_spec_data_train = h5f['train'][:2] # Loads the first 2 samples
    log_mel_spec_data_labels = h5f['label'][:2] 

In [5]:
HOP_LENGTH = 256
SAMPLE_RATE = 22050
VAE = load_model("/home/chua/projects/Autoencoder Notes/Interpretable Sound Generation/initial_full_vae_log_mel_spec.keras",
custom_objects={
    "VariationalAutoencoder": VariationalAutoencoder,
    "Encoder": EncoderBuilder,
    "Decoder": DecoderBuilder,
    "Sampling" : Sampling
})
sound_generator = SoundGenerator(VAE, HOP_LENGTH, SAMPLE_RATE)
signals = sound_generator.generate(log_mel_spec_data_train, "log_mel_spec")
print(type(signals))
print(np.array(signals).shape)

2025-08-20 01:36:47.823055: I external/local_xla/xla/service/service.cc:163] XLA service 0x74f37400dd70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-08-20 01:36:47.823100: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2025-08-20 01:36:47.875061: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-08-20 01:36:48.046221: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90300
2025-08-20 01:36:48.392168: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_247', 12 bytes spill stores, 12 bytes spill loads

2025-08-20 01:36:48.538089: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas w

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


I0000 00:00:1755625011.694153   38905 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


<class 'list'>
(2, 16128)


In [6]:
def save_signal(signals, labels, sample_rate, save_dir='output_wav'):
    os.makedirs(save_dir, exist_ok=True)
    for idx, (signal, label) in enumerate(zip(signals, labels)):
        file_name = f"{label}_{idx}.wav"
        save_path = os.path.join(save_dir, file_name)
        sf.write(save_path, signal, sample_rate)
        print(f"Saved: {save_path}")

In [7]:
save_signal(signals, log_mel_spec_data_labels, SAMPLE_RATE)

Saved: output_wav/4_0.wav
Saved: output_wav/5_1.wav
